# Recompute per-configuration diagnostics from saved kernel matrices

Loads every saved quantum kernel file in your Drive `qml_project` folder and recomputes,
for each configuration, the exact geometric difference across the full lambda sweep, the
off-diagonal means of the quantum and diagnostic RBF kernels, the proxy geometric
difference, concentration, and both alignments. No quantum computation; CPU only, a few
seconds per file. Output: `diagnostics_all_configs.json`, one entry per kernel file.

In [ ]:
import numpy as np, json, os, glob, re
from sklearn.metrics.pairwise import rbf_kernel
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR='/content/drive/MyDrive/qml_project'

def kta(K,y):
    y=np.asarray(y); n=len(y); Y=np.zeros((n,n))
    for c in np.unique(y): m=(y==c).astype(float); Y+=np.outer(m,m)
    H=np.eye(n)-np.ones((n,n))/n; Kc=H@K@H; Yc=H@Y@H
    return float(np.sum(Kc*Yc)/np.sqrt(np.sum(Kc*Kc)*np.sum(Yc*Yc)))
def concentration(K):
    mu=np.maximum(np.linalg.eigvalsh(K),0); lam=mu/mu.sum()
    return float(np.exp(-np.sum(lam*np.log(lam+1e-15)))/K.shape[0])
def g_exact(KQ,KC,lam=1e-6):
    n=KQ.shape[0]; KQ=KQ*n/np.trace(KQ); KC=KC*n/np.trace(KC)
    w,V=np.linalg.eigh(KQ); w=np.maximum(w,0); sq=(V*np.sqrt(w))@V.T
    return float(np.sqrt(max(np.linalg.eigvalsh(sq@np.linalg.inv(KC+lam*n*np.eye(n))@sq).max(),0.0)))
def g_proxy(KQ,KC):
    eff=lambda K:(np.trace(K)**2)/(np.sum(K**2)+1e-10); return float(eff(KQ)/(eff(KC)+1e-10))

files=sorted(set(glob.glob(os.path.join(SAVE_DIR,'*kernel*.npz'))))
print(f"found {len(files)} kernel files")
out={}
for f in files:
    z=np.load(f); keys=set(z.files)
    kK=next((k for k in ['K_train_quantum','K_train','K_q_train'] if k in keys),None)
    kX=next((k for k in ['X_tr','X_train','X_tr_subset'] if k in keys),None)
    kY=next((k for k in ['y_tr','y_train','y_tr_subset'] if k in keys),None)
    if not (kK and kX and kY):
        print(f"  skip {os.path.basename(f)}: keys={sorted(keys)}"); continue
    K=z[kK].astype(np.float64); X=z[kX].astype(np.float64); y=z[kY]
    if K.shape[0]!=1000: print(f"  note {os.path.basename(f)}: N={K.shape[0]}")
    Kc=rbf_kernel(X,X,gamma=1.0/X.shape[1]); off=~np.eye(K.shape[0],dtype=bool)
    tag=re.sub(r'_?kernel.*$','',os.path.basename(f)).replace('.npz','') or os.path.basename(f)
    out[os.path.basename(f)]={'tag_guess':tag,'n':int(K.shape[0]),
        'kta':kta(K,y),'c_kta':kta(Kc,y),'concentration':concentration(K),
        'g_exact':g_exact(K,Kc),'g_proxy':g_proxy(K,Kc),
        'g_lambda_sweep':{f"{l:.0e}":g_exact(K,Kc,l) for l in [1e-4,1e-5,1e-6,1e-7]},
        'offdiag_quantum':float(K[off].mean()),'offdiag_rbf':float(Kc[off].mean())}
    print(f"  {os.path.basename(f):<45} n={K.shape[0]} A_Q={out[os.path.basename(f)]['kta']:.4f} off_q={out[os.path.basename(f)]['offdiag_quantum']:.4f} off_rbf={out[os.path.basename(f)]['offdiag_rbf']:.4f}")
p=os.path.join(SAVE_DIR,'diagnostics_all_configs.json'); json.dump(out,open(p,'w'),indent=2)
print(f"\nsaved {p} with {len(out)} entries. Upload it.")